In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import itertools
import copy
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1.  DATASET CLASS DEFINITION
# ==========================================
class SyntheticMaaSDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)
        
        df['Age_norm'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()
        df['Income_norm'] = (df['Income'] - df['Income'].mean()) / df['Income'].std()
        
        self.z = torch.tensor(df[['Age_norm', 'Income_norm']].values, dtype=torch.float32)
        self.x = torch.tensor(df[['Time_Car', 'Cost_Car', 'Time_PT', 'Cost_PT', 'Time_MaaS', 'Cost_MaaS']].values, dtype=torch.float32)
        self.choice = torch.tensor(df['Choice'].values, dtype=torch.long)
        
        self.ind1 = torch.tensor(df['Ind_1'].values - 1, dtype=torch.long)
        self.ind2 = torch.tensor(df['Ind_2'].values - 1, dtype=torch.long)
        self.ind3 = torch.tensor(df['Ind_3'].values - 1, dtype=torch.long)

    def __len__(self):
        return len(self.choice)

    def __getitem__(self, idx):
        return self.x[idx], self.z[idx], self.choice[idx], self.ind1[idx], self.ind2[idx], self.ind3[idx]

# ==========================================
# 2. MODEL ARCHITECTURE: BI-DNN (Full Mediation)
# ==========================================
class BI_DNN(nn.Module):
    def __init__(self, x_dim=6, z_dim=2, hidden_dim=20, ind_levels=5, num_choices=3):
        super(BI_DNN, self).__init__()
        
        self.latent_net = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1) 
        )
        
        self.ind1_net = nn.Linear(1, ind_levels)
        self.ind2_net = nn.Linear(1, ind_levels)
        self.ind3_net = nn.Linear(1, ind_levels)
        
        self.choice_net = nn.Sequential(
            nn.Linear(x_dim + 1, hidden_dim), 
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_choices)
        )

    def forward(self, x, z):
        lv = self.latent_net(z)
        ind1_logits = self.ind1_net(lv)
        ind2_logits = self.ind2_net(lv)
        ind3_logits = self.ind3_net(lv)
        
        choice_input = torch.cat((x, lv), dim=1) 
        choice_logits = self.choice_net(choice_input)
        
        return choice_logits, ind1_logits, ind2_logits, ind3_logits, lv

# ==========================================
# 3. MULTI-TASK LOSS FUNCTION
# ==========================================
def calculate_loss(choice_logits, ind1_logits, ind2_logits, ind3_logits, 
                   choice_target, ind1_target, ind2_target, ind3_target, lambda_ind=1.0):
    criterion = nn.CrossEntropyLoss()
    loss_choice = criterion(choice_logits, choice_target)
    
    loss_ind1 = criterion(ind1_logits, ind1_target)
    loss_ind2 = criterion(ind2_logits, ind2_target)
    loss_ind3 = criterion(ind3_logits, ind3_target)
    loss_indicator = (loss_ind1 + loss_ind2 + loss_ind3) / 3.0
    
    total_loss = loss_choice + lambda_ind * loss_indicator
    return total_loss, loss_choice

# ==========================================
# 4. TRAINING ENGINE
# ==========================================
def run_single_training(train_loader, test_loader, hidden_dim, lr, epochs=100, seed=42, verbose=False):
    """Executes a single training session and returns metrics of the best-performing model (by NLL)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = BI_DNN(hidden_dim=hidden_dim).to(device)   
    optimizer = optim.Adam(model.parameters(), lr=lr)
    choice_criterion = nn.CrossEntropyLoss(reduction='sum') 
    
    best_nll = float('inf')
    best_acc = 0.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        for x, z, choice, ind1, ind2, ind3 in train_loader:
            x, z = x.to(device), z.to(device)
            choice, ind1, ind2, ind3 = choice.to(device), ind1.to(device), ind2.to(device), ind3.to(device)
            
            optimizer.zero_grad()
            choice_logits, ind1_logits, ind2_logits, ind3_logits, _ = model(x, z)
            
            loss, _ = calculate_loss(choice_logits, ind1_logits, ind2_logits, ind3_logits, 
                                     choice, ind1, ind2, ind3, lambda_ind=1.0)
            loss.backward()
            optimizer.step()
            
        # Validation Phase
        model.eval()
        correct, total_samples, total_nll = 0, 0, 0.0
        with torch.no_grad():
             for tx, tz, tc, _, _, _ in test_loader:
                 tx, tz, tc = tx.to(device), tz.to(device), tc.to(device)
                 t_logits, _, _, _, _ = model(tx, tz)
                 t_preds = torch.argmax(t_logits, dim=1)
                 
                 correct += (t_preds == tc).sum().item()
                 total_nll += choice_criterion(t_logits, tc).item()
                 total_samples += tc.size(0)
                 
        current_acc = correct / total_samples
        current_nll = total_nll / total_samples
        if current_nll < best_nll:
            best_nll = current_nll
            best_acc = current_acc
            best_model_state = copy.deepcopy(model.state_dict())
            
    if verbose:
        print(f"  [Seed {seed}] Best NLL: {best_nll:.4f}, Best Acc: {best_acc*100:.2f}%")
        
    return best_nll, best_acc, best_model_state

# ==========================================
# 5. ROBUSTNESS EVALUATION
# ==========================================
def run_robustness_evaluation(csv_file='synthetic_maas_dataset.csv', batch_size=200):
    print(f"Starting Robustness Evaluation on {device}...\n")
    
    dataset = SyntheticMaaSDataset(csv_file)
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    np.random.seed(42) 
    np.random.shuffle(indices) 
    split = int(np.floor(0.2 * dataset_size))
    train_indices, test_indices = indices[split:], indices[:split]

    train_sampler = torch.utils.data.SubsetRandomSampler(train_indices)
    train_loader = DataLoader(dataset, batch_size=batch_size, sampler=train_sampler)
    test_dataset = torch.utils.data.Subset(dataset, test_indices)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # ---------------------------------------------------------
    # 50 Independent Runs 
    # ---------------------------------------------------------
    print("=== 50 Independent Runs for Robustness ===")
    num_runs = 50
    results_nll = []
    results_acc = []
    
   
    absolute_best_nll = float('inf')
    absolute_best_state = None
    
    for i in range(num_runs):
        seed = i 
        nll, acc, model_state = run_single_training(train_loader, test_loader, 
                                                    hidden_dim=20, 
                                                    lr=0.001, 
                                                    epochs=100, seed=seed, verbose=True)
        results_nll.append(nll)
        results_acc.append(acc)
        
        if nll < absolute_best_nll:
            absolute_best_nll = nll
            absolute_best_state = copy.deepcopy(model_state)

    mean_nll, std_nll = np.mean(results_nll), np.std(results_nll)
    mean_acc, std_acc = np.mean(results_acc) * 100, np.std(results_acc) * 100
    
    print("\n==============================================")
    print("=== FINAL ROBUSTNESS RESULTS (Over 50 Runs) ===")
    print("==============================================")
    print(f"Model: BI-DNN (Full Mediation)")
    print(f"Test Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")
    print(f"Test NLL:      {mean_nll:.4f} ± {std_nll:.4f}")
    print("==============================================")
    
    torch.save(absolute_best_state, 'bc_dnn_synthetic_best_run_overall.pth')
    print("The weights of the single best run have been saved to 'bc_dnn_synthetic_best_run_overall.pth'.")

if __name__ == "__main__":
    run_robustness_evaluation()

Starting Robustness Evaluation on cuda...

=== 50 Independent Runs for Robustness ===
  [Seed 0] Best NLL: 0.7628, Best Acc: 64.80%
  [Seed 1] Best NLL: 0.7645, Best Acc: 65.95%
  [Seed 2] Best NLL: 0.7634, Best Acc: 65.50%
  [Seed 3] Best NLL: 0.7650, Best Acc: 65.45%
  [Seed 4] Best NLL: 0.7613, Best Acc: 65.45%
  [Seed 5] Best NLL: 0.7678, Best Acc: 65.45%
  [Seed 6] Best NLL: 0.7635, Best Acc: 65.45%
  [Seed 7] Best NLL: 0.7639, Best Acc: 64.90%
  [Seed 8] Best NLL: 0.7635, Best Acc: 65.15%
  [Seed 9] Best NLL: 0.7645, Best Acc: 65.00%
  [Seed 10] Best NLL: 0.7665, Best Acc: 66.05%
  [Seed 11] Best NLL: 0.7619, Best Acc: 65.20%
  [Seed 12] Best NLL: 0.7564, Best Acc: 65.15%
  [Seed 13] Best NLL: 0.7647, Best Acc: 65.70%
  [Seed 14] Best NLL: 0.7623, Best Acc: 65.90%
  [Seed 15] Best NLL: 0.7616, Best Acc: 64.25%
  [Seed 16] Best NLL: 0.7698, Best Acc: 65.40%
  [Seed 17] Best NLL: 0.7627, Best Acc: 65.70%
  [Seed 18] Best NLL: 0.7662, Best Acc: 65.90%
  [Seed 19] Best NLL: 0.7627, B